# IPW Effect Estimation and Post-Weighting Balance

## Scenario
Apply inverse probability weighting (IPW) to estimate the causal effect of the 'first box free' trial offer on user conversion and first-month revenue. Verify that IPW successfully reduces covariate imbalance, and compare adjusted estimates to naive estimates to quantify the impact of selection bias.

## Your task
1. IPW weights computed from propensity scores
2. An SMD table comparing balance **before and after** IPW
3. IPW ATE for **both** `converted_to_paid` and `first_month_revenue`
4. Naive ATE for both outcomes (for comparison)
5. A written credibility assessment

## Data: `mealkit_trial_adoption.csv`

**IPW weight formula:**
- Treated units: `weight = 1 / propensity`
- Control units: `weight = 1 / (1 − propensity)`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('Exercise_Data')
if not DATA_DIR.exists():
    DATA_DIR = Path('../Exercise_Data')

CHOICE_PATH = DATA_DIR / 'mealkit_choice_tasks.csv'
PLANS_PATH  = DATA_DIR / 'mealkit_plans.csv'
ADOPT_PATH  = DATA_DIR / 'mealkit_trial_adoption.csv'
SURVEY_PATH = DATA_DIR / 'mealkit_survey.csv'
MMM_PATH    = DATA_DIR / 'mealkit_marketing_weekly.csv'
from sklearn.linear_model import LogisticRegression

In [ ]:
# Refit propensity model — same setup as the propensity overlap exercise
# (This cell is pre-filled so you can proceed directly to IPW estimation)
df = pd.read_csv(ADOPT_PATH)
df_enc = df.fillna({'region': 'Unknown'})
df_enc = pd.get_dummies(df_enc, columns=['region','device'], drop_first=True)
encoded_cat_cols = [c for c in df_enc.columns if c.startswith(('region_','device_'))]
numeric_covs = ['age','household_size','prior_orders','weekly_app_sessions']
all_covs = numeric_covs + encoded_cat_cols
X = df_enc[all_covs].astype(float)
y = df_enc['trial_offer_sent']
prop_model = LogisticRegression(max_iter=500, random_state=42)
prop_model.fit(X, y)
df_enc['propensity'] = prop_model.predict_proba(X)[:, 1]
print(f"Propensity range: {df_enc['propensity'].min():.4f} – {df_enc['propensity'].max():.4f}")

In [ ]:
# ── Step 1: Compute IPW weights ──────────────────────────────────────────
# TODO: Use np.where to assign weights:
#       Treated (trial_offer_sent == 1): weight = 1 / propensity
#       Control (trial_offer_sent == 0): weight = 1 / (1 - propensity)
#       Store as df_enc['ipw']
# TODO: Print df_enc['ipw'].describe().round(3) to check for extreme weights

In [ ]:
# ── Step 2: Post-IPW balance check (SMD before and after) ────────────────
# TODO: Separate df_enc into treated and control groups
# TODO: Write two SMD functions:
#       smd_raw(col, treated, control)  — unweighted SMD
#       smd_ipw(col, df_full)           — weighted SMD using df_enc['ipw']
#       For weighted SMD: use np.average(values, weights=weights)
# TODO: Compute before and after SMD for all 4 numeric covariates
# TODO: Build a DataFrame with columns ['SMD Before', 'SMD After (IPW)'] and print
# TODO: Print max |SMD| before and after

In [ ]:
# ── Step 3: Naive and IPW ATE — both outcomes ────────────────────────────
# TODO: Write two ATE functions:
#       naive_ate(outcome, treated, control) — simple mean difference
#       ipw_ate(outcome, treated, control)   — weighted mean difference using df_enc['ipw']
# TODO: Compute and print naive ATE and IPW ATE for:
#       (1) 'converted_to_paid'  — report in percentage points (multiply by 100)
#       (2) 'first_month_revenue' — report in $/user
# TODO: For each outcome, also print: Difference (naive − IPW) as 'selection bias'

## Credibility Assessment

**Did SMD improve after IPW?**
Reference the max |SMD| before and after weighting.

[YOUR ANSWER]

**What is the magnitude of selection bias (naive − IPW ATE)?**
What does this gap tell you about the direction of targeting — were targeted users more or less likely to convert regardless of the offer?

[YOUR ANSWER]

**Which estimate would you present to a stakeholder, and why?**
Include one sentence acknowledging the remaining unconfoundedness assumption.

[YOUR ANSWER]
